In [4]:
import geopandas as gpd

epa = gpd.read_file("/work/hawkins_lab/vulcan/data/gis/epa-sld/smartlocationdb_4326.shp", engine="pyogrio", use_arrow=True)

# Convert all column names to lowercase
epa.columns = epa.columns.str.lower()
# Filter out anything outside the continental US
epa = epa[(epa.statefp.astype(int) <=56)]
# Make a unique cbsa value by state for areas outside cbsa using county code
# If a CBG is not in a CBSA, then use the county as the equivalent of the CBSA to get aggregate areal statistics
epa.loc[epa.cbsa.isna(),"cbsa"] = epa.loc[epa.cbsa.isna(),"statefp"].astype(int) + epa.loc[epa.cbsa.isna(),"countyfp"].astype(int)/100
epa["cbsa"] = round(epa["cbsa"].astype(float),2)
epa.loc[epa.d4a==-99999, 'd4a'] = epa.d4a.replace(-99999,0)

In [5]:
list(epa.columns)

['objectid',
 'geoid10',
 'geoid20',
 'statefp',
 'countyfp',
 'tractce',
 'blkgrpce',
 'csa',
 'csa_name',
 'cbsa',
 'cbsa_name',
 'cbsa_pop',
 'cbsa_emp',
 'cbsa_wrk',
 'ac_total',
 'ac_water',
 'ac_land',
 'ac_unpr',
 'totpop',
 'counthu',
 'hh',
 'p_wrkage',
 'autoown0',
 'pct_ao0',
 'autoown1',
 'pct_ao1',
 'autoown2p',
 'pct_ao2p',
 'workers',
 'r_lowwagew',
 'r_medwagew',
 'r_hiwagewk',
 'r_pctlowwa',
 'totemp',
 'e5_ret',
 'e5_off',
 'e5_ind',
 'e5_svc',
 'e5_ent',
 'e8_ret',
 'e8_off',
 'e8_ind',
 'e8_svc',
 'e8_ent',
 'e8_ed',
 'e8_hlth',
 'e8_pub',
 'e_lowwagew',
 'e_medwagew',
 'e_hiwagewk',
 'e_pctlowwa',
 'd1a',
 'd1b',
 'd1c',
 'd1c5_ret',
 'd1c5_off',
 'd1c5_ind',
 'd1c5_svc',
 'd1c5_ent',
 'd1c8_ret',
 'd1c8_off',
 'd1c8_ind',
 'd1c8_svc',
 'd1c8_ent',
 'd1c8_ed',
 'd1c8_hlth',
 'd1c8_pub',
 'd1d',
 'd1_flag',
 'd2a_jphh',
 'd2b_e5mix',
 'd2b_e5mixa',
 'd2b_e8mix',
 'd2b_e8mixa',
 'd2a_ephhm',
 'd2c_trpmx1',
 'd2c_trpmx2',
 'd2c_tripeq',
 'd2r_jobpop',
 'd2r_wrkemp',
 

In [6]:
# Function to calculate weighted sum
def weighted_avg(values, weights):
    return (values * weights.sum(axis=1)).sum() / weights.sum(axis=1).sum()

grp_col = "cbsa"

epa_cbsa = epa.drop_duplicates(subset=grp_col).copy()

# Group by CBSA and calculate sum for cols
val_cols = ["ac_total","ac_water","ac_land","ac_unpr","totpop","counthu","hh",
            "autoown0","autoown1","autoown2p","workers","r_lowwagew","r_medwagew",
           "r_hiwagewk","totemp","e5_ret","e5_off","e5_ind","e5_svc","e5_ent","e8_ret",
            "e8_off","e8_ind","e8_svc","e8_ent","e8_ed","e8_hlth","e8_pub","e_lowwagew",
           "e_medwagew","e_hiwagewk"]
for val_col in val_cols:
    epa_cbsa.loc[:, val_col] = epa.groupby(grp_col).apply(lambda x: sum(x[val_col]), include_groups=False
    ).values

# Calculated from other SLD columns
auto_own_cols = ["autoown0","autoown1","autoown2p"]
epa_cbsa.loc[:, "pct_ao0"] = epa_cbsa["autoown0"] / (epa_cbsa[auto_own_cols].sum(axis=1))
epa_cbsa.loc[:, "pct_ao1"] = epa_cbsa["autoown1"] / (epa_cbsa[auto_own_cols].sum(axis=1))
epa_cbsa.loc[:, "pct_ao2p"] = epa_cbsa["autoown2p"] / (epa_cbsa[auto_own_cols].sum(axis=1))

wage_cols = ["r_lowwagew","r_medwagew","r_hiwagewk"]
epa_cbsa["r_pctlowwage"] = epa_cbsa.loc[:,"r_lowwagew"] / (epa_cbsa.loc[:,wage_cols].sum(axis=1))

emp_cols = ["e_lowwagew","e_medwagew","e_hiwagewk"]
epa_cbsa["e_pctlowwage"] = epa_cbsa.loc[:,"e_lowwagew"] / (epa_cbsa.loc[:,emp_cols].sum(axis=1))

# density_cols = ["d1a","d1b","d1c","d1c5_ret","d1c5_off","d1c5_ind","d1c5_svc",
#                "d1c5_ent","d1c8_ret","d1c8_off","d1c8_ind","d1c8_svc","d1c8_ent",
#                 "d1c8_ed","d1c8_hlth","d1c8_pub"]
# pop_cols = ["counthu","e8_ret","e8_off","e8_ind","e8_svc","e8_ent","e8_ed","e8_hlth","e8_pub"]
# area_cols = ["ac_unpr","ac_total"]
# area_indicator = "d1_flag"
# for d_col, p_col in zip(density_cols, pop_cols):
#     if area_indicator==1:
#         epa.loc[:, d_col] = epa.loc[:, p_col] / epa.loc[:, area_cols[1]]
#     else:
#         epa.loc[:, d_col] = epa.loc[:, p_col] / epa.loc[:, area_cols[0]]

# Group by CBSA and calculate weighted avg for cols by population
wgt_cols = ["totpop"]
val_cols = ["p_wrkage","d2a_jphh","d4a","d4e","d5ar","d5br","d2a_ranked","d2b_ranked",
           "d3b_ranked","d4a_ranked","natwalkind", "d4c","d1a","d1b"]
for val_col in val_cols:
    epa_cbsa.loc[:, val_col] = epa.groupby(grp_col).apply(lambda x: weighted_avg(x[val_col], x.loc[:, wgt_cols]), include_groups=False
    ).values
    
# Group by CBSA and calculate weighted avg for cols by employment
wgt_cols = ["totemp"]
val_cols = ["d2b_e5mix","d2b_e5mixa","d2b_e8mix","d2b_e8mixa","d2a_wrkemp","d2c_wremlx",
           "d4b025","d4b050","d5ae","d5be","d1c","d1c5_ret","d1c5_off","d1c5_ind","d1c5_svc",
               "d1c5_ent","d1c8_ret","d1c8_off","d1c8_ind","d1c8_svc","d1c8_ent",
                "d1c8_ed","d1c8_hlth","d1c8_pub"]
for val_col in val_cols:
    epa_cbsa.loc[:, val_col] = epa.groupby(grp_col).apply(lambda x: weighted_avg(x[val_col], x.loc[:, wgt_cols]), include_groups=False
    ).values
    
# Group by CBSA and calculate weighted avg for cols by population+employment
wgt_cols = ["totpop","totemp"]
val_cols = ["d2a_ephhm","d2c_trpmx1","d2c_trpmx2","d2c_tripeq","d2r_jobpop"]
for val_col in val_cols:
    epa_cbsa.loc[:, val_col] = epa.groupby(grp_col).apply(lambda x: weighted_avg(x[val_col], x.loc[:, wgt_cols]), include_groups=False
    ).values
    
# Group by CBSA and calculate weighted avg for cols by unprotected land area
wgt_cols = ["ac_unpr"]
val_cols = ["d3a","d3aao","d3amm","d3apo","d3b","d3bao","d3bmm3","d3bmm4","d3bpo3","d3bpo4",
            "d4d"]
for val_col in val_cols:
    epa_cbsa.loc[:, val_col] = epa.groupby(grp_col).apply(lambda x: weighted_avg(x[val_col], x.loc[:, wgt_cols]), include_groups=False
    ).values

In [7]:
# Add suffix to all column names
suffix = '_cbsa'
epa_cbsa.columns = [col + suffix for col in epa_cbsa.columns]

inc_cols = ["cbsa","ac_total","ac_water","ac_land","ac_unpr","totpop","counthu","hh",
            "autoown0","autoown1","autoown2p","workers","r_lowwagew","r_medwagew",
           "r_hiwagewk","totemp","e5_ret","e5_off","e5_ind","e5_svc","e5_ent","e8_ret",
            "e8_off","e8_ind","e8_svc","e8_ent","e8_ed","e8_hlth","e8_pub","e_lowwagew",
           "e_medwagew","e_hiwagewk","pct_ao0","pct_ao1","pct_ao2p","r_pctlowwage","e_pctlowwage",
           "d1a","d1b","d1c","d1c5_ret","d1c5_off","d1c5_ind","d1c5_svc",
            "d1c5_ent","d1c8_ret","d1c8_off","d1c8_ind","d1c8_svc","d1c8_ent",
            "d1c8_ed","d1c8_hlth","d1c8_pub","counthu","e8_ret","e8_off","e8_ind","e8_svc","e8_ent","e8_ed","e8_hlth","e8_pub",
           "p_wrkage","d2a_jphh","d4a","d4e","d5ar","d5br","d2a_ranked","d2b_ranked",
           "d3b_ranked","d4a_ranked","natwalkind", "d4c","d2b_e5mix","d2b_e5mixa","d2b_e8mix","d2b_e8mixa","d2a_wrkemp","d2c_wremlx",
           "d4b025","d4b050","d5ae","d5be","d2a_ephhm","d2c_trpmx1","d2c_trpmx2","d2c_tripeq","d2r_jobpop",
           "d3a","d3aao","d3amm","d3apo","d3b","d3bao","d3bmm3","d3bmm4","d3bpo3","d3bpo4","d4d"]

inc_cols = [col + suffix for col in inc_cols]

epa_cbsa.to_csv("/work/hawkins_lab/vulcan/data/transport/epa_cbsa.csv", index=False, columns=inc_cols)

In [8]:
epa.d1a.describe()

count    217739.000000
mean          4.525163
std          13.938563
min           0.000000
25%           0.295216
50%           1.841697
75%           4.140478
max        1481.257220
Name: d1a, dtype: float64

In [9]:
epa_cbsa.d1a_cbsa.describe()

count    2045.000000
mean        0.770882
std         1.507940
min         0.000153
25%         0.138831
50%         0.405677
75%         0.965283
max        37.973778
Name: d1a_cbsa, dtype: float64